### Sample 10% of videos from outputs folder

In [2]:
import os
import random
import math

In [ ]:


base_dir = '/Users/eveyhuang/Documents/NICO/gemini_code/outputs'

# Collect all JSON files and map them to their video file names
video_file_map = {}
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith('.json') and not file.startswith('all_') and not file.startswith('verbal_'):
            video_name = file.replace('.json', '')
            full_path = os.path.join(root, file)
            if video_name not in video_file_map:
                video_file_map[video_name] = []
            video_file_map[video_name].append(full_path)

# Get all unique video names
all_video_names = list(video_file_map.keys())
print(f"Total unique video names found: {len(all_video_names)}")
n_sample = max(1, math.ceil(0.1 * len(all_video_names)))  # At least 1

print(f"Number of videos to sample: {n_sample}")
# Randomly sample 10% of video names
random.seed(42)  # For reproducibility
sampled_video_names = random.sample(all_video_names, n_sample)

# Get all file paths for the sampled videos
sampled_file_paths = []
for name in sampled_video_names:
    sampled_file_paths.extend(video_file_map[name])

# Print or save the sampled file paths
print("Sampled JSON files for verification:")
for path in sampled_file_paths:
    print(path)

# Optionally, save to a text file
with open('sampled_json_files.txt', 'w') as f:
    for path in sampled_file_paths:
        f.write(path + '\n')

Total unique video names found: 781
Number of videos to sample: 79
Sampled JSON files for verification:
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MND/output_2021_04_22_MND_S6/Breakout_Room_4_Part_2_2021_04_22_13_14_53/Breakout_Room_4_Part_2_2021_04_22_13_14_53_chunk6.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021ABI/output_2021_05_21_ABI_S5/bot5p3_Room_5_Zoom_Meeting_5_21_2021_10_59_19_AM/bot5p3_Room_5_Zoom_Meeting_5_21_2021_10_59_19_AM_chunk3.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021ABI/output_2021_05_20_ABI_S4/bot2p2_Zoom_Meeting_2021_05_20_12_37_07/bot2p2_Zoom_Meeting_2021_05_20_12_37_07_chunk3.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MZT/output_2021_10_01_MZT_S1/B1.2_Zoom_Meeting_Room_1_2021_10_01_11_07_45/B1.2_Zoom_Meeting_Room_1_2021_10_01_11_07_45.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021SLU/output_2021_06_10_SLU_S5/botB2_2021_06_10_12_35_06/botB2_2021_06_10_12_35_06_chunk2.json
/Users/eveyh

In [9]:
import json
import os
import re
sampled_dict = {}

with open('sampled_json_files.txt', 'r') as f:
    file_paths = [line.strip() for line in f if line.strip()]

def extract_key_and_subkey(path):
    # Get the part after '/outputs/'
    rest = path.split('/outputs/')[1]
    parts = rest.split(os.sep)
    key = os.path.join(parts[0], parts[1])
    sub_key = os.path.join(*parts[2:])  # Join everything after the key
    return key, sub_key

def find_verbal_annotations(data, file):
    for dt in file:
        if dt["transcript"] == data["transcript"]:
            data["annotations"] = dt["annotations"]
    return data


for path in file_paths:
    # Extract key after 'output_'
    try:
        key, sub_key = extract_key_and_subkey(path)
        
    except Exception as e:
        print(f"Skipping {path}: {e}")
        continue
    
    folder, filename = os.path.split(path)
    filename = re.sub(r'_chunk\d+(?=\.json)', '', filename)
    # Load JSON and sample a value
    try:
        with open(path, 'r') as jf:
            data = json.load(jf)
        with open(os.path.join(folder, 'all_'+filename), 'r') as f:
            all_data = json.load(f)
        if isinstance(data, list) and data:
            sampled_value = random.choice(data)
        elif isinstance(data, dict) and data:
            filtered = [ann for ann in data["meeting_annotations"] if ann["speaking duration"] > 15]
            if filtered:
                sample_size = min(3, len(filtered))
                sampled_value = random.sample(filtered, sample_size)
                for sample in sampled_value:
                    sample = find_verbal_annotations(sample, all_data)
            else:
                # Fallback to any annotation if none meet the criteria
                sampled_value = None
        else:
            sampled_value = data
    except Exception as e:
        print(f"Error reading {path}: {e}")
        continue

    if key not in sampled_dict:
        sampled_dict[key] = {}
    sampled_dict[key][sub_key] = sampled_value

# Save to a new JSON file
with open('sampled_verification.json', 'w') as out_f:
    json.dump(sampled_dict, out_f, indent=2)

Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MZT/output_2021_10_01_MZT_S1/B1.1_Zoom_Meeting_Room_1_2021_10_01_11_04_18/B1.1_Zoom_Meeting_Room_1_2021_10_01_11_04_18.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S3/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44_chunk3.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S3/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44_chunk4.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S6/6_Theory_and_Expt_Zoom_Meeting_2020_11_05_10_28_39/6_Theory_and_Expt_Zoom_Meeting_2020_11_05_10_28_39_chunk1.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_06_NES_S7/1_beyond_co2

In [10]:
import pandas as pd

# Flatten sampled_verification for DataFrame
rows = []
for folder, files in sampled_dict.items():
    for file, samples in files.items():
        # samples can be a list or None
        if samples is None:
            rows.append({'folder': folder, 'file': file})
        elif isinstance(samples, list):
            for sample in samples:
                row = {'folder': folder, 'file': file}
                if isinstance(sample, dict):
                    row.update(sample)
                rows.append(row)
        elif isinstance(samples, dict):
            row = {'folder': folder, 'file': file}
            row.update(samples)
            rows.append(row)
        else:
            row = {'folder': folder, 'file': file, 'value': samples}
            rows.append(row)

df = pd.DataFrame(rows)
df.to_excel('sampled_verification.xlsx', index=False)

# Importing Evey's request

In [1]:
from pathlib import Path
BASE_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/outputs").expanduser()
RANDOM_SEED = 3839
OUT_CSV = BASE_DIR.parent / "sampling" / "sample_balanced.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [2]:
def find_all_gm_v4_files(base_dir: Path):
    return sorted(base_dir.rglob("all_gm_v4*.json"))

files = find_all_gm_v4_files(BASE_DIR)
print("Files found:", len(files))
assert files, "No all_gm_v4*.json files found under BASE_DIR; double-check the path."

Files found: 213


In [14]:
# === 10% FILE-LEVEL SAMPLING IMPLEMENTATION ===

from pathlib import Path
import json, pandas as pd, random, math, re
from datetime import datetime
from collections import Counter

# --- Setup ---
BASE_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/outputs").expanduser()
RANDOM_SEED = 3839
OUT_CSV_FILES = BASE_DIR.parent / "sampling" / "sample_10pct_files.csv"
OUT_CSV_FILES.parent.mkdir(parents=True, exist_ok=True)

# --- Helpers ---
def find_all_gm_v4_files(base_dir: Path):
    return sorted(base_dir.rglob("all_gm_v4*.json"))

def load_json_records(fp: Path):
    try:
        with fp.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list): return data
        if isinstance(data, dict):
            for v in data.values():
                if isinstance(v, list): return v
    except json.JSONDecodeError:
        pass
    # Fallback: NDJSON
    recs = []
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict): recs.append(obj)
            except json.JSONDecodeError:
                continue
    return recs

def extract_codes_from_item(item: dict):
    for key in ("labels", "codes"):
        if key in item and isinstance(item[key], list):
            return [str(x) for x in item[key] if isinstance(x, (str, int, float))]
    ann = item.get("annotations"); codes = []
    if isinstance(ann, dict):
        for k, v in ann.items():
            if isinstance(v, (bool, int, float)):
                if bool(v): codes.append(str(k))
            elif isinstance(v, dict):
                if "score" in v:
                    try:
                        if float(v["score"]) > 0: codes.append(str(k))
                    except Exception:
                        if bool(v["score"]): codes.append(str(k))
                elif "value" in v:
                    try:
                        if float(v["value"]) > 0: codes.append(str(k))
                    except Exception:
                        if bool(v["value"]): codes.append(str(k))
                elif any(bool(x) for x in v.values()):
                    codes.append(str(k))
    return codes

def infer_conference_from_path(p: Path, anchor="outputs") -> str:
    parts = list(p.parts)
    return parts[parts.index(anchor)+1] if anchor in parts and parts.index(anchor)+1 < len(parts) else "unknown"

def build_dataframe(file_paths):
    rows = []
    for fp in file_paths:
        for i, rec in enumerate(load_json_records(fp)):
            rows.append({
                "conference": infer_conference_from_path(fp),
                "json_file": str(fp),
                "record_idx": i,
                "codes": extract_codes_from_item(rec),
                "speaker": rec.get("speaker"),
                "timestamp": rec.get("timestamp"),
                "transcript": rec.get("transcript"),
                "speaking_duration_raw": rec.get("speaking duration"),
                "start_time": rec.get("start_time"),
                "end_time": rec.get("end_time"),
            })
    return pd.DataFrame(rows)

# Duration parsing
def _parse_hhmm_or_mmss_to_seconds(s: str) -> float:
    if not isinstance(s, str) or ":" not in s: return math.nan
    parts = s.strip().split(":")
    try: nums = list(map(int, parts))
    except Exception: return math.nan
    if len(nums) == 2: mm, ss = nums; return mm*60 + ss
    if len(nums) == 3: hh, mm, ss = nums; return hh*3600 + mm*60 + ss
    return math.nan

def normalize_duration_seconds(row) -> float:
    v = row.get("speaking_duration_raw")
    if isinstance(v, (int, float)): return float(v)
    if isinstance(v, str):
        v = v.strip()
        if re.fullmatch(r"\d+(\.\d+)?", v): return float(v)
        sec = _parse_hhmm_or_mmss_to_seconds(v)
        if sec == sec: return sec
    start, end = row.get("start_time"), row.get("end_time")
    if isinstance(start, str) and isinstance(end, str) and ":" in start and ":" in end:
        try:
            fmt = "%H:%M:%S" if len(start.split(":")) == 3 else "%H:%M"
            t0, t1 = datetime.strptime(start, fmt), datetime.strptime(end, fmt)
            delta = (t1 - t0).total_seconds()
            if delta >= 0: return float(delta)
        except Exception:
            pass
    return math.nan

# --- Pipeline ---
files = find_all_gm_v4_files(BASE_DIR)
print("Total gm_v4 files:", len(files))

rng = random.Random(RANDOM_SEED)
K = max(1, round(0.10 * len(files)))
sampled_files = rng.sample(files, K)

print(f"Sampling 10% of files → {K} files")
print("Example files:", [f.name for f in sampled_files[:3]])

df_files = build_dataframe(sampled_files)
print("Utterances loaded from sampled files:", len(df_files))

# Duration calculation
df_files["duration_sec"] = df_files.apply(normalize_duration_seconds, axis=1)

# Filters
MIN_DURATION_SEC = 15
df_files = df_files[df_files["codes"].apply(lambda lst: isinstance(lst, list) and len(lst) > 0)].copy()
df_files["codes"] = df_files["codes"].apply(lambda lst: [c for c in lst if c != "None"])
df_files = df_files[df_files["codes"].apply(lambda lst: len(lst) > 0)].copy()
df_files = df_files[df_files["duration_sec"].fillna(0) >= MIN_DURATION_SEC].copy()

print("After filters — utterances from sampled files:", len(df_files))
print("Unique utterances (videos to watch):", len(df_files))

# Per-code + per-conference summary
per_code_files = Counter(c for lst in df_files["codes"] for c in lst)
print("\nPer-code counts in the 10%-files sample:")
for k in sorted(per_code_files):
    print(f"{k}: {per_code_files[k]}")

print("\nConferences represented:", sorted(df_files["conference"].unique()))
print("\nPer-conference row counts (10%-files sample):")
print(df_files["conference"].value_counts().sort_index())

# Save
df_files.to_csv(OUT_CSV_FILES, index=False)
print("Wrote 10%-of-files sample to:", OUT_CSV_FILES)

Total gm_v4 files: 213
Sampling 10% of files → 21 files
Example files: ['all_gm_v4_bot3p2_Zoom_Meeting_2021_05_20_12_40_54.json', 'all_gm_v4_B3_2021_11_05_11_01_14_trimend.json', 'all_gm_v4_Bot_6_Zoom_Meeting_Room_6_2021_05_20_12_36_53.json']
Utterances loaded from sampled files: 2556
After filters — utterances from sampled files: 865
Unique utterances (videos to watch): 865

Per-code counts in the 10%-files sample:
Coordination and Decision Practices: 137
Evaluation Practices: 137
Idea Management: 330
Information Seeking: 214
Integration Practices: 71
Knowledge Sharing: 604
Participation Dynamics: 81
Relational Climate: 153

Conferences represented: ['2020NES', '2021ABI', '2021CMC', '2021MND', '2021MZT', '2021NES', '2022MND']

Per-conference row counts (10%-files sample):
conference
2020NES    105
2021ABI    132
2021CMC     79
2021MND     57
2021MZT    121
2021NES    224
2022MND    147
Name: count, dtype: int64
Wrote 10%-of-files sample to: /Users/maxchalekson/Desktop/gemini_data_anal